In [1]:
###### import os # Allows for files to be downloaded to local directory
import requests # Manages HTTP requests to the Figshare webpage, making communication easier
from concurrent.futures import ThreadPoolExecutor, as_completed # Downloads multiple files at once

# Set working directory
os.chdir(r"C:\Users\Alexei Clark\Documents\depmap_test")

# Function for single file download
# Used to clarify to ThreadPoolExecuter that single files are to be downloaded
def download_single_file(file_meta, download_dir):

    # Save each file by name, web url, and direct it to the local directory
    filename = file_meta['name']
    download_url = file_meta['download_url']
    file_path = os.path.join(download_dir, filename)

    # 5-second connect timeout, 60-second read timeout
    # Used for troubleshooting if the script is unable to stop running
    with requests.get(download_url, stream=True, timeout=(5, 60)) as file_res:
        file_res.raise_for_status()
        
        # Write file chunks directly to disk, with an 8Kb chunk-size to improve accessibility
        with open(file_path, "wb") as f:
            for chunk in file_res.iter_content(chunk_size=8192):
                f.write(chunk)

    # Identifies that individual file is successfully downloaded
    return f"Successfully downloaded: {filename}"

# Create function to update DepMap files
def update_depmap_files(release_query, download_dir="depmap_data", max_workers=6):

    # Make new directory (depmap_data) for downloads
    os.makedirs(download_dir, exist_ok=True)

    # Identify Figshare API from the Figshare database URL
    print(f"\nQuerying Figshare API for '{release_query}'...")
    search_url = "https://api.figshare.com/v2/articles/search"

    # Search the specified database for user query
    search_payload = {
        "search_for": f':title: "{release_query}"'
    }

    # Search request with timeout to avoid hanging indefinitely
    print("Sending search query to Figshare...")
    response = requests.post(search_url, json=search_payload, timeout=(5, 30))
    response.raise_for_status()

    # Determine if the query, consisting of many files, is identified
    articles = response.json()

    # Stops file if prior condition is untrue
    if not articles:
        print(f"No releases found matching '{release_query}'. Please check the format and try again.")
        return

    # Searches for the latest release (First file) of the appropriate article ID and release
    # Multiple metrics are used to ensure that the EXACT file is identified
    latest_release = articles[0]
    article_id = latest_release['id']
    print(f"Release detected: {latest_release['title']}")

    # Grabs the url of the user query
    details_url = f"https://api.figshare.com/v2/articles/{article_id}"
    print("Fetching article file list details...")

    # Standard mechanism for identifying a status
    details_response = requests.get(details_url, timeout=(5, 30))
    details_response.raise_for_status()

    # Assuming no errors, stores the file data in JSON format
    all_files = details_response.json().get('files', [])
    print(f"Discovered {len(all_files)} total files. Initiating concurrent download. '{max_workers}' files are downloaded concurrently.")
    
    # Uses ThreadPoolExecutor to download files in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:

        # Identifies that all files are subject to executor
        future_to_file = {
            executor.submit(download_single_file, file_meta, download_dir): file_meta['name'] 
            for file_meta in all_files
        }
        
        # Process results as they complete
        # Uses the Future object to ensure concurrent downloads, which are stored into the result_message variable
        for future in as_completed(future_to_file):
            result_message = future.result()
            print(result_message)
                    
    print(f"\nSuccessfully synchronized all {len(all_files)} files!")

# Prompt to enter a file name
if __name__ == "__main__":
    print("=====================================================")
    print("              DepMap File Download Tool              ")
    print("=====================================================")
    print("Standard Format: 'DepMap YYQX Public'")
    print("YY = Year (Last 2 Digits) and X = An integer 1-4")
    print("Note: The standard format must be typed exactly")
    print("or the script will fail to run.")
    print("-----------------------------------------------------")
    print("Example User Query: DepMap 21Q1 Public")
    print("=====================================================\n")

    # user_query is stored and is equated to response_query
    user_query = input("Enter the DepMap release name you want to download: ").strip()

    # Prompt user for number of concurrent workers between 4 and 8.
    print("Enter the number of preferred download workers as an integer between 4 and 8.")
    print("Higher integers will allow for more concurrent downloads,")
    print("though at the expense of computer performance.")
    workers_input = input("Please input a number: ").strip()

    # Checks if input is greater than or equal to 4 and less than or equal to 6
    if workers_input.isdigit() and 4 <= int(workers_input) <= 8:
        max_workers = int(workers_input)

    # If an invalid input is entered, then the max_workers will be declared as 6
    else:
        if workers_input:
            print("Invalid input or out of range (4-8). Using default of 6 workers.")
        max_workers = 6

    # Updates the max workers in user query
    if user_query:
        update_depmap_files(release_query=user_query, max_workers=max_workers)

              DepMap File Download Tool              
Standard Format: 'DepMap YYQX Public'
YY = Year (Last 2 Digits) and X = An integer 1-4
Note: The standard format must be typed exactly
or the script will fail to run.
-----------------------------------------------------
Example User Query: DepMap 21Q1 Public



Enter the DepMap release name you want to download:  DepMap 20Q1 Public


Enter the number of preferred download workers as an integer between 4 and 8.
Higher integers will allow for more concurrent downloads,
though at the expense of computer performance.


Please input a number:  6



Querying Figshare API for 'DepMap 20Q1 Public'...
Sending search query to Figshare...
Release detected: DepMap 20Q1 Public
Fetching article file list details...
Discovered 24 total files. Initiating concurrent download. '6' files are downloaded concurrently.
Successfully downloaded: Achilles_high_variance_genes.csv
Successfully downloaded: Achilles_dropped_guides.csv
Successfully downloaded: Achilles_common_essentials.csv
Successfully downloaded: Achilles_replicate_map.csv
Successfully downloaded: Achilles_guide_map.csv
Successfully downloaded: Achilles_guide_efficacy.csv
Successfully downloaded: CCLE_fusions.csv
Successfully downloaded: CCLE_fusions_unfiltered.csv
Successfully downloaded: CCLE_expression.csv
Successfully downloaded: CCLE_segment_cn.csv
Successfully downloaded: common_essentials.csv
Successfully downloaded: nonessentials.csv
Successfully downloaded: sample_info.csv
Successfully downloaded: CCLE_expression_full.csv
Successfully downloaded: README
Successfully downloade